# GPUs for Deep Learning

Companion notebook for the [GPUs for Deep Learning lesson](https://ml-viz-ruby.vercel.app/courses/gpu-programming/04-gpus-for-deep-learning).

We make the lesson's claims quantitative: why **matmul** has high arithmetic intensity (and grows
more compute-bound with size), how **mixed precision** halves memory traffic, how **kernel fusion**
collapses memory round-trips, and why **batching** rescues memory-bound LLM decode. Pure NumPy.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#444', 'axes.labelcolor': '#ccc',
    'xtick.color': '#888', 'ytick.color': '#888',
    'text.color': '#eee', 'grid.color': '#333', 'lines.linewidth': 2,
})

## 1 — Matmul intensity grows with size

A square `N x N` matmul costs `2N^3` FLOPs and moves `~3 * N^2 * bytes_per_elem` bytes (two inputs +
one output). So arithmetic intensity `I = FLOPs / bytes` grows linearly with `N` — large matmuls
drift firmly into compute-bound territory, the GPU's strong suit.

In [ ]:
def matmul_flops(M, N, K):
    return 2 * M * N * K

def matmul_intensity(N, bytes_per_elem=4):
    flops = matmul_flops(N, N, N)
    bytes_moved = 3 * N * N * bytes_per_elem        # A, B, C
    return flops / bytes_moved

sizes = np.array([16, 32, 64, 128, 256, 512, 1024, 2048, 4096])
I = [matmul_intensity(n) for n in sizes]

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.loglog(sizes, I, 'o-', color='#2dd4bf')
ax.axhline(20, ls='--', color='#fb7185', label='example ridge point (20 FLOP/byte)')
ax.set_xlabel('matrix size N'); ax.set_ylabel('arithmetic intensity (FLOP/byte)')
ax.set_title('Bigger matmuls are more compute-bound (intensity grows with N)')
ax.legend(facecolor='#1a1d27', edgecolor='#444'); ax.grid(True, alpha=0.3, which='both')
plt.tight_layout(); plt.show()

print(f"N=64   intensity = {matmul_intensity(64):.1f} FLOP/byte")
print(f"N=4096 intensity = {matmul_intensity(4096):.0f} FLOP/byte  (deeply compute-bound)")

## 2 — Mixed precision halves the bytes

Doing matmuls in 16-bit (FP16/BF16) instead of FP32 halves the bytes moved for the same FLOPs,
**doubling** arithmetic intensity for a bandwidth-limited op — and engaging tensor cores on top.

In [ ]:
N = 1024
fp32 = matmul_intensity(N, bytes_per_elem=4)
fp16 = matmul_intensity(N, bytes_per_elem=2)
print(f"FP32 intensity: {fp32:.0f} FLOP/byte")
print(f"FP16 intensity: {fp16:.0f} FLOP/byte  ({fp16/fp32:.1f}x higher -> better for memory-bound regimes)")

# activations/gradients memory for a hidden state
elems = 8 * 2048 * 4096        # batch x seq x hidden
print(f"\nactivation tensor: {elems/1e6:.0f}M elements")
print(f"  FP32: {elems*4/1e9:.2f} GB")
print(f"  FP16: {elems*2/1e9:.2f} GB  (half the memory and bandwidth)")

## 3 — Kernel fusion collapses memory round-trips

A chain of `k` element-wise ops each reads its input from global memory and writes its output back:
`2k` round-trips. Fusing them does one load, all the arithmetic in registers, one store: `2` trips.
For these memory-bound chains, that approaches a `k`x speedup.

In [ ]:
def roundtrips_unfused(k):
    return 2 * k

def roundtrips_fused(k):
    return 2

for k in [1, 2, 3, 5]:
    u, f = roundtrips_unfused(k), roundtrips_fused(k)
    print(f"chain of {k} ops:  unfused={u} trips  fused={f} trips  -> {u/f:.1f}x less memory traffic")

# numeric demo: bias -> relu -> scale, fused vs staged, same result
x = np.random.default_rng(0).normal(size=10000).astype(np.float32)
bias, scale = 0.5, 2.0
staged = x + bias; staged = np.maximum(staged, 0); staged = staged * scale
fused = np.maximum(x + bias, 0) * scale          # one pass, no intermediates stored
assert np.allclose(staged, fused)
print("\n\u2713 fused result matches staged result (fusion changes traffic, not math)")

## 4 — Batching rescues memory-bound decode

Generating one token re-reads the whole weight matrix to produce a single output column — tiny
intensity. Batching `B` requests reuses each loaded weight across `B` tokens, so intensity scales
with the batch size, pushing decode from memory-bound toward compute-bound.

In [ ]:
def decode_intensity(batch, d_model=4096, bytes_per_elem=2):
    # weight matrix d_model x d_model applied to a batch of vectors
    flops = 2 * batch * d_model * d_model
    bytes_moved = (d_model * d_model + 2 * batch * d_model) * bytes_per_elem  # weights + in/out
    return flops / bytes_moved

batches = [1, 2, 4, 8, 16, 32, 64, 128, 256]
Ib = [decode_intensity(b) for b in batches]

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.semilogx(batches, Ib, 'o-', color='#818cf8', base=2)
ax.axhline(20, ls='--', color='#fb7185', label='example ridge (20 FLOP/byte)')
ax.set_xlabel('batch size'); ax.set_ylabel('arithmetic intensity (FLOP/byte)')
ax.set_title('Batching raises decode intensity: memory-bound -> compute-bound')
ax.legend(facecolor='#1a1d27', edgecolor='#444'); ax.grid(True, alpha=0.3, which='both')
plt.tight_layout(); plt.show()

print(f"batch 1:   intensity = {decode_intensity(1):.2f} FLOP/byte (memory-bound, GPU idle)")
print(f"batch 256: intensity = {decode_intensity(256):.0f} FLOP/byte (compute-bound, GPU busy)")

## ✏️ Your turn

**Exercise.** Implement `fusion_speedup(k)` — the factor by which fusing a chain of `k` memory-bound
element-wise ops reduces global-memory traffic (round-trips unfused ÷ fused) — and
`bytes_saved(k, tensor_bytes)`, the global-memory bytes saved per pass given each op's tensor is
`tensor_bytes` (unfused moves `2k * tensor_bytes`; fused moves `2 * tensor_bytes`).

In [ ]:
def fusion_speedup(k):
    # TODO(you): unfused round-trips / fused round-trips
    return ...

def bytes_saved(k, tensor_bytes):
    # TODO(you): (unfused trips - fused trips) * tensor_bytes
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
assert fusion_speedup(1) == 1.0           # nothing to fuse
assert fusion_speedup(5) == 5.0           # 10 trips -> 2 trips
assert bytes_saved(1, 1_000_000) == 0     # single op saves nothing
assert bytes_saved(3, 1_000_000) == 4_000_000   # (6 - 2) * 1MB
assert matmul_flops(128, 512, 256) == 33_554_432
print("\u2713 fusion + FLOP accounting checks pass")

<details>
<summary>Solution</summary>

```python
def fusion_speedup(k):
    return (2 * k) / 2          # = k

def bytes_saved(k, tensor_bytes):
    return (2 * k - 2) * tensor_bytes
```

Fusion turns `2k` global round-trips into `2`, a `k`x reduction for memory-bound chains. This is
exactly what `torch.compile`, XLA, and FlashAttention do — and why the giant attention score matrix
never needs to touch global memory.

</details>